# Charts for thesis

## Classification

### Confusion matrix

In [ ]:
from pyexpat import model

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Load predictions using pathlib
preds_path = Path("test_results") / "Classification" / "predictions" / "test_preds_100k_tuned.parquet"
df_preds = pd.read_parquet(preds_path)

model_name = "LightGBM"
y_true = df_preds["true_y"]
y_pred_proba = df_preds[model_name]

# Optimal threshold calculation (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

# Class predictions using the optimal threshold
y_pred_class = (y_pred_proba >= optimal_threshold).astype(int)

# Confusion matrix elements
tn, fp, fn, tp = confusion_matrix(y_true, y_pred_class).ravel()

# Dynamically calculate cell colors using matplotlib's "Reds" colormap
cmap = plt.get_cmap("Reds")
vmax = max(tn, fp, fn, tp)

# Multiply vmax to ensure the darkest color is light enough for clean black text
norm = mcolors.Normalize(vmin=0, vmax=vmax * 2.6)

def get_cell_color(val):
    # Extracts the hex code without the alpha channel
    return mcolors.to_hex(cmap(norm(val)))[1:].upper()

hex_tn, hex_fp, hex_fn, hex_tp = map(get_cell_color, [tn, fp, fn, tp])

# Generate the native LaTeX TikZ code with a clean, modern aesthetic
latex_code = f"""
\\begin{{figure}}[htbp]
\\centering
\\begin{{tikzpicture}}[
    cell/.style={{rectangle, minimum width=3.2cm, minimum height=3.2cm, align=center, font=\\sffamily, text=black}},
    axisl/.style={{font=\\sffamily\\bfseries}},
    axis_val/.style={{font=\\sffamily\\bfseries, text=black}}
]
    \\definecolor{{colorTN}}{{HTML}}{{{hex_tn}}}
    \\definecolor{{colorFP}}{{HTML}}{{{hex_fp}}}
    \\definecolor{{colorFN}}{{HTML}}{{{hex_fn}}}
    \\definecolor{{colorTP}}{{HTML}}{{{hex_tp}}}

    \\node[cell, fill=colorTN] (TN) at (0, 0) {{\\Large TN\\\\[0.2cm] \\large {tn}}};
    \\node[cell, fill=colorFP] (FP) at (3.3, 0) {{\\Large FP\\\\[0.2cm] \\large {fp}}};
    \\node[cell, fill=colorFN] (FN) at (0, -3.3) {{\\Large FN\\\\[0.2cm] \\large {fn}}};
    \\node[cell, fill=colorTP] (TP) at (3.3, -3.3) {{\\Large TP\\\\[0.2cm] \\large {tp}}};

    \\node[axisl] at (1.65, 2.3) {{Predikovaná třída}};
    \\node[axisl, rotate=90] at (-2.3, -1.65) {{Skutečná třída}};
    
    \\node[axis_val] at (0, 1.8) {{0}};
    \\node[axis_val] at (3.3, 1.8) {{1}};
    \\node[axis_val] at (-1.8, 0) {{0}};
    \\node[axis_val] at (-1.8, -3.3) {{1}};

    \\path (5.6, 0);

\\end{{tikzpicture}}
\\caption{{Matice záměn -- {model_name} (100k dataset). Optimální hranice: {optimal_threshold:.4f}.}}
\\label{{fig:matice_zamen_{model_name.lower()}_100k}}
\\end{{figure}}
"""

print(latex_code)


\begin{figure}[htbp]
\centering
\begin{tikzpicture}[
    cell/.style={rectangle, minimum width=3.2cm, minimum height=3.2cm, align=center, font=\sffamily, text=black},
    axisl/.style={font=\sffamily\bfseries}, % Ponecháno pro nadpisy (které šablona obarví modře)
    axis_val/.style={font=\sffamily\bfseries, text=black} % Explicitně vynucená černá pro hodnoty 0 a 1
]
    \definecolor{colorTN}{HTML}{FC8F6F}
    \definecolor{colorFP}{HTML}{FCC1A8}
    \definecolor{colorFN}{HTML}{FFEDE5}
    \definecolor{colorTP}{HTML}{FEE1D4}

    \node[cell, fill=colorTN] (TN) at (0, 0) {\Large TN\\[0.2cm] \large 131821};
    \node[cell, fill=colorFP] (FP) at (3.3, 0) {\Large FP\\[0.2cm] \large 79237};
    \node[cell, fill=colorFN] (FN) at (0, -3.3) {\Large FN\\[0.2cm] \large 16602};
    \node[cell, fill=colorTP] (TP) at (3.3, -3.3) {\Large TP\\[0.2cm] \large 40948};

    % Nadpisy (převezmou modrou barvu z vlastností vaší Latex šablony/hooku)
    \node[axisl] at (1.65, 2.3) {Predikovaná třída};
    \n

### ROC Curve

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score

# Load data and probabilities for models from the 100k tuned dataset
preds_path = Path("test_results") / "Classification" / "predictions" / "test_preds_100k_tuned.parquet"
df_preds = pd.read_parquet(preds_path)

y_true = df_preds["true_y"]
model_name = "LightGBM"
y_pred_proba = df_preds[model_name]

# Calculate ROC curve and AUC score for LightGBM
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
auc_score = roc_auc_score(y_true, y_pred_proba)

# Downsample points to approx ~200 to prevent pdflatex memory limits
step = max(1, len(fpr) // 500)
fpr_reduced = fpr[::step]
tpr_reduced = tpr[::step]

# Ensure the curve naturally ends at exactly (1, 1)
if fpr_reduced[-1] != 1.0 or tpr_reduced[-1] != 1.0:
    fpr_reduced = np.append(fpr_reduced, 1.0)
    tpr_reduced = np.append(tpr_reduced, 1.0)
    
coords = "\n        ".join([f"({f:.4f}, {t:.4f})" for f, t in zip(fpr_reduced, tpr_reduced)])

# Generate the native LaTeX PGFPlots code
latex = f"""\\begin{{figure}}[htbp]
\\centering
\\begin{{tikzpicture}}[trim axis left, trim axis right] % Dokonalé vycentrování osy grafu
\\begin{{axis}}[
    width=8cm,
    height=8cm,
    scale only axis=true,
    title={{\\textbf{{ROC křivka -- {model_name} (AUC = {auc_score:.4f})}}}},
    xlabel={{False Positive Rate}},
    ylabel={{True Positive Rate}},
    xmin=-0.02, xmax=1.02,
    ymin=-0.02, ymax=1.02,
    grid=major,
    grid style={{dashed, gray!30}},
    enlargelimits=false,
    tick label style={{font=\\sffamily}},
    label style={{font=\\sffamily\\bfseries}},
    title style={{font=\\sffamily\\bfseries}}
]

    \\addplot [
        color=black!40!green,
        ultra thick,
        solid
    ] coordinates {{
        {coords}
    }};

\\end{{axis}}
\\end{{tikzpicture}}
\\caption{{ROC křivka modelu {model_name} trénovaného na datové sadě o 100 tisících vzorcích, vyhodnoceno na testovací množině.}}
\\label{{fig:roc_auc_100k_lgbm}}
\\end{{figure}}
"""

print(latex)

\begin{figure}[htbp]
\centering
\begin{tikzpicture}[trim axis left, trim axis right] % Dokonalé vycentrování osy grafu
\begin{axis}[
    width=8cm,
    height=8cm,
    scale only axis=true,
    title={\textbf{ROC křivka -- LightGBM (AUC = 0.7314)}},
    xlabel={False Positive Rate},
    ylabel={True Positive Rate},
    xmin=-0.02, xmax=1.02,
    ymin=-0.02, ymax=1.02,
    grid=major,
    grid style={dashed, gray!30},
    enlargelimits=false,
    tick label style={font=\sffamily},
    label style={font=\sffamily\bfseries},
    title style={font=\sffamily\bfseries}
]

    \addplot [
        color=black!40!green,
        ultra thick,
        solid
    ] coordinates {
        (0.0000, 0.0000)
        (0.0005, 0.0054)
        (0.0011, 0.0093)
        (0.0017, 0.0139)
        (0.0023, 0.0179)
        (0.0029, 0.0218)
        (0.0035, 0.0255)
        (0.0041, 0.0289)
        (0.0048, 0.0326)
        (0.0055, 0.0360)
        (0.0061, 0.0395)
        (0.0067, 0.0427)
        (0.0073, 0.0465)
  

### Results

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths to the score directories
scores_dir = Path("test_results") / "Classification" / "scores"
datasets = ["1k", "10k", "100k", "full"]

# Data structure [dataset][model_base]
roc_auc_data = {ds: {} for ds in datasets}
all_models = set()

# Load data into dictionaries
for ds in datasets:
    file = scores_dir / f"test_scores_{ds}_tuned.csv"
    if not file.exists():
        continue
        
    df = pd.read_csv(file)
    for _, row in df.iterrows():
        model_raw = row["Model"]
        
        # Exclude Dummy model entirely
        if "Dummy" in model_raw:
            continue
        
        # Determine if the model was trained on a smaller dataset or not tuned
        has_asterisk = "(not_tuned)" in model_raw or "(10k)" in model_raw or "(100k)" in model_raw or "(30k)" in model_raw
        model_base = model_raw.split(" (")[0]
        
        all_models.add(model_base)
        
        # Store as a tuple: (value, boolean_has_asterisk)
        if has_asterisk:
            roc_auc_data[ds][model_base] = (None, True)
        else:
            roc_auc_data[ds][model_base] = (row["ROC AUC"], False)

# Sort models alphabetically to ensure visual consistency
all_models = sorted(list(all_models))

# Function to generate the transposed LaTeX table with booktabs formatting
def generate_latex_table_transposed(data_dict, title, label):
    # Příprava hlaviček
    header_models = " & ".join([f"\\textbf{{{m}}}" for m in all_models])
    header_empty = " & ".join(["" for _ in all_models])
    
    latex = f"""\\begin{{table}}[htbp]
\\centering
\\small
\\setlength{{\\tabcolsep}}{{4.5pt}}
\\renewcommand{{\\arraystretch}}{{1.25}}
\\begin{{tabular}}{{l {' c' * len(all_models)}}}
\\toprule
\\textbf{{Počet záznamů}} & {header_models} \\\\[-0.3em]
\\textbf{{trénovací sady}} & {header_empty} \\\\
\\midrule
"""
    
    # Najdeme nejlepší model pro každou datovou sadu, abychom ho mohli zvýraznit
    best_per_ds = {}
    for ds in datasets:
        valid_vals = []
        for m in all_models:
            if m in data_dict[ds]:
                val, is_ast = data_dict[ds][m]
                if not is_ast and val is not None:
                    valid_vals.append(val)
        
        if valid_vals:
            best_per_ds[ds] = max(valid_vals)  # Pro ROC AUC hledáme maximum
        else:
            best_per_ds[ds] = None

    has_global_asterisk = False
    
    # Generování řádků přes datové sady
    for ds in datasets:
        row_str = f"\\textbf{{{ds}}} "
        for model in all_models:
            if model in data_dict[ds]:
                val, is_ast = data_dict[ds][model]
                
                if is_ast:
                    has_global_asterisk = True
                    row_str += "& $^*$ "
                elif val is not None:
                    # Česká desetinná čárka zapouzdřená do závorek pro LaTeX
                    num_str = f"{val:.4f}".replace(".", "{,}")
                    
                    # Zvýraznění nejlepší buňky (na úrovni dané datové sady)
                    if best_per_ds[ds] is not None and abs(val - best_per_ds[ds]) < 1e-6:
                        row_str += f"& \\cellcolor{{green!25}}\\textbf{{{num_str}}} "
                    else:
                        row_str += f"& {num_str} "
                else:
                    row_str += "& - "
            else:
                row_str += "& - "
                
        row_str += "\\\\\n"
        latex += row_str
        
    latex += "\\bottomrule\n\\end{tabular}\n"
    
    # Poznámka pod čarou, pokud existují chybějící hodnoty
    if has_global_asterisk:
        footnote_msg = "U těchto modelů nebylo možné z časových důvodů provést ladění hyperparametrů na plné datové sadě (GBM, NGBoost), nebo model neumožňuje trénování na takto velkých datech (TabPFN)."
        latex += f"\\\\ {{\\footnotesize $^*$ {footnote_msg}}}\n"
        
    latex += f"\\caption{{{title}}}\n\\label{{{label}}}\n\\end{{table}}\n"
    return latex

print(generate_latex_table_transposed(
    roc_auc_data, 
    "ROC AUC skóre modelů napříč trénovacími sadami vyhodnocené na stejné testovací množině.", 
    "tab:roc_auc_scores"
))

% -- TRANSPOSED ROC AUC TABLE --
\begin{table}[htbp]
\centering
% -- Zúžení tabulky --
\small
\setlength{\tabcolsep}{4.5pt}
\renewcommand{\arraystretch}{1.25}
% --------------------
\begin{tabular}{l  c c c c c c c}
\toprule
\textbf{Počet záznamů} & \textbf{CatBoost} & \textbf{GBM} & \textbf{HistGBM} & \textbf{LightGBM} & \textbf{NGBoost} & \textbf{TabPFN} & \textbf{XGBoost} \\[-0.3em]
\textbf{trénovací sady} &  &  &  &  &  &  &  \\
\midrule
\textbf{1k} & 0{,}6794 & 0{,}6853 & 0{,}6767 & 0{,}6837 & 0{,}6806 & \cellcolor{green!25}\textbf{0{,}6984} & 0{,}6688 \\
\textbf{10k} & 0{,}7062 & 0{,}7109 & 0{,}7102 & 0{,}7137 & 0{,}7118 & \cellcolor{green!25}\textbf{0{,}7177} & 0{,}7117 \\
\textbf{100k} & 0{,}7298 & 0{,}7223 & 0{,}7284 & \cellcolor{green!25}\textbf{0{,}7315} & 0{,}7304 & 0{,}7302 & 0{,}7305 \\
\textbf{full} & 0{,}7260 & $^*$ & 0{,}7272 & 0{,}7286 & $^*$ & $^*$ & \cellcolor{green!25}\textbf{0{,}7300} \\
\bottomrule
\end{tabular}
\\ {\footnotesize $^*$ U těchto modelů nebylo možné

## Regression

### Results

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

scores_dir = Path("test_results") / "Regression" / "scores"
datasets = ["1k", "10k", "100k", "full"]

rmse_data = {ds: {} for ds in datasets}
all_models = set()

for ds in datasets:
    file = scores_dir / f"test_scores_{ds}_tuned.csv"
    if not file.exists():
        continue
        
    df = pd.read_csv(file)
    for _, row in df.iterrows():
        model_raw = row["Model"]
        
        if "Dummy" in model_raw:
            continue
        
        has_asterisk = "(not_tuned)" in model_raw or "(10k)" in model_raw or "(100k)" in model_raw or "(30k)" in model_raw
        model_base = model_raw.split(" (")[0]
        
        all_models.add(model_base)
        
        if has_asterisk:
            rmse_data[ds][model_base] = (None, True)
        else:
            rmse_data[ds][model_base] = (row["RMSE"], False)

all_models = sorted(list(all_models))

def generate_latex_table_rmse_transposed(data_dict, title, label):
    header_models = " & ".join([f"\\textbf{{{m}}}" for m in all_models])
    header_empty = " & ".join(["" for _ in all_models])
    
    latex = f"""\\begin{{table}}[htbp]
\\centering
\\resizebox{{\\textwidth}}{{!}}{{
\\small
\\setlength{{\\tabcolsep}}{{3.5pt}}
\\renewcommand{{\\arraystretch}}{{1.25}}
\\begin{{tabular}}{{l {' c' * len(all_models)}}}
\\toprule
\\textbf{{Počet záznamů}} & {header_models} \\\\[-0.3em]
\\textbf{{trénovací sady}} & {header_empty} \\\\
\\midrule
"""
    
    best_per_ds = {}
    for ds in datasets:
        valid_vals = []
        for m in all_models:
            if m in data_dict[ds]:
                val, is_ast = data_dict[ds][m]
                if not is_ast and val is not None:
                    valid_vals.append(val)
        
        if valid_vals:
            best_per_ds[ds] = min(valid_vals)
        else:
            best_per_ds[ds] = None

    has_global_asterisk = False
    
    for ds in datasets:
        row_str = f"\\textbf{{{ds}}} "
        for model in all_models:
            if model in data_dict[ds]:
                val, is_ast = data_dict[ds][model]
                
                if is_ast:
                    has_global_asterisk = True
                    row_str += "& $^*$ "
                elif val is not None:
                    num_str = f"{val:.4f}".replace(".", "{,}")
                    
                    if best_per_ds[ds] is not None and abs(val - best_per_ds[ds]) < 1e-6:
                        row_str += f"& \\cellcolor{{green!25}}\\textbf{{{num_str}}} "
                    else:
                        row_str += f"& {num_str} "
                else:
                    row_str += "& - "
            else:
                row_str += "& - "
                
        row_str += "\\\\\n"
        latex += row_str
        
    latex += "\\bottomrule\n\\end{tabular}\n"
    latex += "}\n"
    
    if has_global_asterisk:
        footnote_msg = "U těchto modelů nebylo proveditelné z časových důvodů provést ladění nebo trénování na plné sadě (GBM, NGBoost, TabPFN)."
        latex += f"\\vspace{{0.1cm}}\\\\ {{\\footnotesize $^*$ {footnote_msg}}}\n"
        
    latex += f"\\caption{{{title}}}\n\\label{{{label}}}\n\\end{{table}}\n"
    return latex


print(generate_latex_table_rmse_transposed(
    rmse_data, 
    "RMSE skóre regresních modelů napříč trénovacími sadami vyhodnocené na stejné testovací množině.", 
    "tab:rmse_scores"
))

% -- TRANSPOSED RMSE TABLE --
\begin{table}[htbp]
\centering
\resizebox{\textwidth}{!}{
\small
\setlength{\tabcolsep}{3.5pt}
\renewcommand{\arraystretch}{1.25}
\begin{tabular}{l  c c c c c c c c}
\toprule
\textbf{Počet záznamů} & \textbf{CatBoost} & \textbf{GBM} & \textbf{HistGBM} & \textbf{LightGBM} & \textbf{NGBoost} & \textbf{PGBM} & \textbf{TabPFN} & \textbf{XGBoost} \\[-0.3em]
\textbf{trénovací sady} &  &  &  &  &  &  &  &  \\
\midrule
\textbf{1k} & 0{,}3613 & 0{,}3631 & 0{,}3628 & 0{,}3622 & 0{,}3624 & 0{,}3637 & \cellcolor{green!25}\textbf{0{,}3609} & 0{,}3620 \\
\textbf{10k} & 0{,}3590 & 0{,}3601 & 0{,}3602 & 0{,}3589 & \cellcolor{green!25}\textbf{0{,}3583} & 0{,}3602 & 0{,}3589 & 0{,}3586 \\
\textbf{100k} & 0{,}3533 & 0{,}3541 & 0{,}3542 & 0{,}3533 & 0{,}3544 & 0{,}3541 & 0{,}3551 & \cellcolor{green!25}\textbf{0{,}3531} \\
\textbf{full} & 0{,}3536 & $^*$ & 0{,}3539 & \cellcolor{green!25}\textbf{0{,}3534} & $^*$ & 0{,}3542 & $^*$ & 0{,}3537 \\
\bottomrule
\end{tabular}
}
\vspac

### PGBM and NGBoost

#### Train the models on optimal parameters and save results

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from ngboost import NGBRegressor
from pgbm.sklearn import HistGradientBoostingRegressor
from ngboost.scores import CRPScore
from ngboost.distns import Normal
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

warnings.filterwarnings("ignore", category=UserWarning, message=".*unknown categories.*")


# 1. Load data
TRAIN_DATA_PATH = Path("data/train_data.parquet")
TEST_DATA_PATH = Path("data/test_data.parquet")

train_df = pd.read_parquet(TRAIN_DATA_PATH)
test_df = pd.read_parquet(TEST_DATA_PATH)

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_reg = train_df["target_annual_roi"]
test_full_X = test_df.drop(["target", "target_annual_roi"], axis=1)
test_full_y_reg = test_df["target_annual_roi"]

train_100k_X = train_full_X.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

# 2. Preprocessing
cols_to_nominal_cat = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()
cardinality = train_df[cols_to_nominal_cat].nunique()
cols_for_ohe = cardinality[cardinality <= 5].index.tolist()
cols_for_te = cardinality[cardinality > 5].index.tolist()

ohe_categories = [train_df[col].dropna().unique().tolist() for col in cols_for_ohe]
ohe_transformer = OneHotEncoder(
    categories=ohe_categories,
    drop="if_binary",
    handle_unknown="ignore",
    sparse_output=False,
)
target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")


# Models and parameters
ngb_base_params = {
    "criterion": "friedman_mse",
    "max_depth": 6,
    "min_samples_split": 370,
    "min_samples_leaf": 55,
    "min_weight_fraction_leaf": 0.023063554740156644,
    "max_features": None,
    "ccp_alpha": 1.5678575150605078e-05,
    "max_leaf_nodes": 231,
    "min_impurity_decrease": 0.9254240780342731,
    "random_state": 42,
}
ngb_base_learner = DecisionTreeRegressor(**ngb_base_params)
ngb_best_params = {
    "n_estimators": 7750,
    "learning_rate": 0.0013414300062166583,
    "minibatch_frac": 0.8153498884932472,
    "col_sample": 0.5413169982792909,
    "random_state": 42,
    "verbose": False,
}
ngb_model = make_pipeline(
    numeric_preprocessor,
    NGBRegressor(Base=ngb_base_learner, Dist=Normal, Score=CRPScore, **ngb_best_params),
)

pgbm_best_params = {
    "max_iter": 942,
    "learning_rate": 0.16852745776554806,
    "max_depth": 19,
    "min_samples_leaf": 400,
    "max_leaf_nodes": 355,
    "l2_regularization": 0.43331171136970614,
    "max_bins": 161,
    "tree_correlation": 0.0002951506064953749,
    "distribution": "logistic",
    "random_state": 42,
    "verbose": 0,
    "with_variance": True,
}
pgbm_model = make_pipeline(
    numeric_preprocessor, HistGradientBoostingRegressor(**pgbm_best_params)
)

# 4. Training & Generating Predictions
preds_path = Path(
    "test_results/Regression/predictions/test_preds_100k_intervals_ngboost_pgbm_tuned.parquet"
)


print("Training NGBoost...")
ngb_model.fit(train_100k_X, train_100k_y_reg)
ngb_dist = ngb_model.named_steps["ngbregressor"].pred_dist(
    numeric_preprocessor.transform(test_full_X)
)
ngb_preds = ngb_dist.loc
ngb_std = ngb_dist.scale
ngb_lower = ngb_preds - 1.96 * ngb_std
ngb_upper = ngb_preds + 1.96 * ngb_std
print("Training PGBM...")
pgbm_model.fit(train_100k_X, train_100k_y_reg)
pgbm_preds, pgbm_std = pgbm_model.named_steps["histgradientboostingregressor"].predict(
    numeric_preprocessor.transform(test_full_X), return_std=True
)
pgbm_lower = pgbm_preds - 1.96 * pgbm_std
pgbm_upper = pgbm_preds + 1.96 * pgbm_std

results_df = pd.DataFrame(
    {
        "y_true": test_full_y_reg.values,
        "ngb_pred": ngb_preds,
        "ngb_std": ngb_std,
        "ngb_lower": ngb_lower,
        "ngb_upper": ngb_upper,
        "pgbm_pred": pgbm_preds,
        "pgbm_std": pgbm_std,
        "pgbm_lower": pgbm_lower,
        "pgbm_upper": pgbm_upper,
    }
)
results_df.to_parquet(preds_path, index=False)

Training NGBoost...
Training PGBM...


In [ ]:
# 5. Generate native TikZ/PGFPlots code for NGBoost intervals (Green style, explicit ticks, legend below)
np.random.seed(42)
sample_idx = np.random.choice(results_df.index, size=50, replace=False)
sample_df = (
    results_df.loc[sample_idx].copy().sort_values(by="y_true").reset_index(drop=True)
)

ngb_pred_coords = []
ngb_true_coords = []
for i, row in sample_df.iterrows():
    ngb_true_coords.append(f"({i}, {row['y_true']:.4f})")
    err = 1.96 * row['ngb_std']
    ngb_pred_coords.append(f"({i}, {row['ngb_pred']:.4f}) +- (0, {err:.4f})")

latex_code = f"""\\begin{{figure}}[htbp]
\\centering
\\begin{{tikzpicture}}[trim axis left, trim axis right]
    \\begin{{axis}}[
        width=0.85\\textwidth,
        height=8cm,
        grid=major,
        grid style={{dashed, gray!30}},
        tick label style={{font=\\sffamily, /pgf/number format/use comma}},
        label style={{font=\\sffamily\\bfseries}},
        title style={{font=\\sffamily\\bfseries}},
        title={{NGBoost -- 95\\% predikční intervaly}},
        ylabel={{Cílová proměnná (ROI)}},
        xtick=\\empty,
        ytick={{-1, -0.5, 0, 0.5}},
        ymin=-1.02, ymax=0.80,
        yticklabels={{-100\\%, -50\\%, 0\\%, 50\\%}},
        legend style={{
            at={{(0.5,-0.10)}},
            anchor=north,
            legend columns=-1,
            nodes={{scale=0.9, transform shape}}, 
            font=\\sffamily
        }},
        enlargelimits=0.05
    ]
        \\addplot [
            only marks,
            mark=*,
            color=black!40!green,
            error bars/.cd,
            y dir=both, y explicit
        ] coordinates {{
            {" ".join(ngb_pred_coords)}
        }};
        \\addlegendentry{{Predikce (95\\% PI)}}
        
        \\addplot [
            only marks,
            mark=x,
            color=red,
            mark size=2.5pt,
            thick
        ] coordinates {{
            {" ".join(ngb_true_coords)}
        }};
        \\addlegendentry{{Skutečná hodnota}}
    \\end{{axis}}
\\end{{tikzpicture}}
\\caption{{95\\% predikční intervaly pro 50 náhodně vybraných testovacích záznamů predikovaných modelem NGBoost trénovaným na datové sadě o $100\\,000$ záznamech. Zobrazené záznamy jsou seřazeny podle skutečné hodnoty cílové proměnné pro lepší vizuální přehled.}}
\\label{{fig:pred_intervals_100k_ngboost}}
\\end{{figure}}
"""

print(latex_code)

\begin{figure}[htbp]
\centering
\begin{tikzpicture}[trim axis left, trim axis right]
    \begin{axis}[
        width=0.85\textwidth,
        height=8cm,
        grid=major,
        grid style={dashed, gray!30},
        tick label style={font=\sffamily, /pgf/number format/use comma},
        label style={font=\sffamily\bfseries},
        title style={font=\sffamily\bfseries},
        title={NGBoost -- 95\% predikční intervaly},
        ylabel={Cílová proměnná (ROI)},
        xtick=\empty,
        ytick={-1, -0.5, 0, 0.5},
        ymin=-1.02, ymax=0.80,
        yticklabels={-100\%, -50\%, 0\%, 50\%},
        legend style={
            at={(0.5,-0.10)},
            anchor=north,
            legend columns=-1,
            nodes={scale=0.9, transform shape}, 
            font=\sffamily
        },
        enlargelimits=0.05
    ]
        \addplot [
            only marks,
            mark=*,
            color=black!40!green,
            error bars/.cd,
            y dir=both, y explicit
    

## Correlation of predictions

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

def generate_combined_latex_corr():
    # Load data
    path_cls = Path("test_results") / "Classification" / "predictions" / "test_preds_100k_tuned.parquet"
    path_reg = Path("test_results") / "Regression" / "predictions" / "test_preds_100k_tuned.parquet"
    
        
    df_cls = pd.read_parquet(path_cls)
    df_reg = pd.read_parquet(path_reg)

    exclude_cols = ["true_y"]
    cols_cls = [c for c in df_cls.columns if "Dummy" not in c and c not in exclude_cols]
    cols_reg = [c for c in df_reg.columns if "Dummy" not in c and c not in exclude_cols]
    
    corr_cls = df_cls[cols_cls].corr(method="spearman")
    corr_reg = df_reg[cols_reg].corr(method="pearson")
    
    dist_matrix = np.clip(1.0 - corr_cls.values, 0, 2)
    np.fill_diagonal(dist_matrix, 0)
    
    Z = hierarchy.linkage(squareform(dist_matrix), method='ward')
    ordered_idx = hierarchy.leaves_list(Z)
    ordered_cls = [cols_cls[i] for i in ordered_idx]
    
    ordered_reg = [c for c in ordered_cls if c in cols_reg]
    extra_models = [c for c in cols_reg if c not in ordered_cls]
    
    extra_models = sorted(extra_models, key=lambda x: 1 if "PGBM" in x else 0)
    ordered_reg.extend(extra_models)
    
    corr_cls = corr_cls.loc[ordered_cls, ordered_cls]
    corr_reg = corr_reg.loc[ordered_reg, ordered_reg]
    
    mask_cls = np.tril(np.ones(corr_cls.shape), k=-1).astype(bool)
    mask_reg = np.tril(np.ones(corr_reg.shape), k=-1).astype(bool)
    
    vmin_cls = corr_cls.where(mask_cls).min().min()
    vmin_reg = corr_reg.where(mask_reg).min().min()
    
    vmin = min(vmin_cls, vmin_reg)
    vmin = np.floor(vmin * 20) / 20.0 
    vmax = 1.0
    
    cmap = plt.get_cmap("Reds")
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    
    def get_color(val):
        intensity = 0.05 + 0.60 * norm(val) 
        return mcolors.to_hex(cmap(intensity))[1:].upper()
        
    top_color = get_color(vmax)
    bottom_color = get_color(vmin)
    
    cell_size = "2.8em"
    v_offset = "-0.9em" 
    
    latex = "\\begin{figure}[htbp]\n\\centering\n"
    latex += "\\setlength{\\tabcolsep}{0pt}\n"
    latex += "\\resizebox{\\textwidth}{!}{\n"
    
    latex += "\\begin{tabular}{c @{\\hspace{0.5cm}} c @{\\hspace{0.4cm}} c}\n"
    
    latex += f"\\renewcommand{{\\arraystretch}}{{1.0}}\n\\begin{{tabular}}[t]{{l *{{{len(ordered_cls)}}}{{c}}}}\n"
    
    num_cols_cls = len(ordered_cls)
    latex += f"\\multicolumn{{1}}{{c}}{{}} & \\multicolumn{{{num_cols_cls}}}{{c}}{{\\makebox[0pt][c]{{\\textbf{{\\Large Spearmanova korelace -- Klasifikace}}}}}} \\\\\n"
    latex += "\\addlinespace[0.2cm]\n"

    header1 = " & " + " & ".join([f"\\makebox[{cell_size}][c]{{\\rotatebox{{45}}{{\\textbf{{{m}}}}}}}" for m in ordered_cls]) + " \\\\\n"
    latex += header1
    
    for m1 in ordered_cls:
        row = f"\\textbf{{{m1}}} \\hspace{{0.5em}} "
        for m2 in ordered_cls:
            val = corr_cls.loc[m1, m2]
            v_str = f"{val:.2f}".replace('.', '{,}')
            hex_col = get_color(val)
            row += f"& \\cellcolor[HTML]{{{hex_col}}}\\makebox[{cell_size}][c]{{\\rule[{v_offset}]{{0pt}}{{{cell_size}}}\\textcolor{{black}}{{{v_str}}}}} "
        row += "\\\\\n"
        latex += row
    latex += "\\end{tabular} &\n"
    
    latex += f"\\renewcommand{{\\arraystretch}}{{1.0}}\n\\begin{{tabular}}[t]{{l *{{{len(ordered_reg)}}}{{c}}}}\n"
    
    num_cols_reg = len(ordered_reg)
    latex += f"\\multicolumn{{1}}{{c}}{{}} & \\multicolumn{{{num_cols_reg}}}{{c}}{{\\makebox[0pt][c]{{\\textbf{{\\Large Pearsonova korelace -- Regrese}}}}}} \\\\\n"
    latex += "\\addlinespace[0.2cm]\n"

    header2 = " & " + " & ".join([f"\\makebox[{cell_size}][c]{{\\rotatebox{{45}}{{\\textbf{{{m}}}}}}}" for m in ordered_reg]) + " \\\\\n"
    latex += header2
    
    for m1 in ordered_reg:
        row = f"\\textbf{{{m1}}} \\hspace{{0.5em}} "
        for m2 in ordered_reg:
            val = corr_reg.loc[m1, m2]
            v_str = f"{val:.2f}".replace('.', '{,}')
            hex_col = get_color(val)
            row += f"& \\cellcolor[HTML]{{{hex_col}}}\\makebox[{cell_size}][c]{{\\rule[{v_offset}]{{0pt}}{{{cell_size}}}\\textcolor{{black}}{{{v_str}}}}} "
        row += "\\\\\n"
        latex += row
    latex += "\\end{tabular} &\n"
    
    latex += f"\\definecolor{{corrTop}}{{HTML}}{{{top_color}}}\n"
    latex += f"\\definecolor{{corrBot}}{{HTML}}{{{bottom_color}}}\n"
    
    latex += "\\raisebox{-9.5cm}{\n\\begin{tikzpicture}\n"
    bar_height = len(ordered_reg) * 0.95
    latex += f"    \\shade[top color=corrTop, bottom color=corrBot] (0,0) rectangle (0.4, {bar_height});\n"
    latex += f"    \\node[right, font=\\small, text=black] at (0.5, {bar_height}) {{1,00}};\n"
    v_str_min = f"{vmin:.2f}".replace('.', '{,}')
    latex += f"    \\node[right, font=\\small, text=black] at (0.5, 0) {{{v_str_min}}};\n"
    latex += "\\end{tikzpicture}\n}\n"
    
    latex += "\\end{tabular}\n}\n"
    
    latex += "\\caption{Korelační matice predikcí modelů na datové sadě o 100 tisících vzorcích. Zobrazeny jsou metriky úměrné metodám ensemblování (Spearmanova korelace pro klasifikaci a Pearsonova pro regresi). Obě matice sdílejí totožné seřazení na ose Y vynucené hierarchickým clusterováním klasifikace a sdílejí jednu plošnou barevnou škálu pro usnadnění vizuálního porovnání.}\n"
    latex += "\\label{fig:corr_matrices_combined_100k}\n"
    latex += "\\end{figure}\n"
    
    return latex

print(generate_combined_latex_corr())

\begin{figure}[htbp]
\centering
\setlength{\tabcolsep}{0pt}
\resizebox{\textwidth}{!}{
\begin{tabular}{c @{\hspace{0.5cm}} c @{\hspace{0.4cm}} c}
\renewcommand{\arraystretch}{1.0}
\begin{tabular}[t]{l *{7}{c}}
\multicolumn{1}{c}{} & \multicolumn{7}{c}{\makebox[0pt][c]{\textbf{\Large Spearmanova korelace -- Klasifikace}}} \\
\addlinespace[0.2cm]
 & \makebox[2.8em][c]{\rotatebox{45}{\textbf{TabPFN}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{GBM}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{CatBoost}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{HistGBM}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{NGBoost}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{XGBoost}}} & \makebox[2.8em][c]{\rotatebox{45}{\textbf{LightGBM}}} \\
\textbf{TabPFN} \hspace{0.5em} & \cellcolor[HTML]{E83429}\makebox[2.8em][c]{\rule[-0.9em]{0pt}{2.8em}\textcolor{black}{1{,}00}} & \cellcolor[HTML]{FB694A}\makebox[2.8em][c]{\rule[-0.9em]{0pt}{2.8em}\textcolor{black}{0{,}96}} & \cellcolor[HTML]{FA6648}\makebox[2.8em][